In [0]:
# ============================================================================
# SUPPLEMENTARY TABLE — Combine entity extraction + LLM topic assignment
# per event (pr_id) grain.
#
# Sources:
#   - us_gmsgq_dev.gms_us_alyt.deviation_embed_input  (entity extraction)
#   - LLM classification (Llama 3.3 70B) for topics
#   - us_gmsgq_dev.gms_us_mart.tw_deviation_data_formatted_rdq (source fields)
# ============================================================================
import re
import json
import pandas as pd
import numpy as np
from pyspark.sql import functions as F, types as T

CATALOG = "us_gmsgq_dev"
ALYT    = "gms_us_alyt"
MART    = "gms_us_mart"

EMBED_INPUT = f"{CATALOG}.{ALYT}.deviation_embed_input"
SOURCE_TABLE = f"{CATALOG}.{MART}.tw_deviation_data_formatted_rdq"
OUTPUT_TABLE = f"{CATALOG}.{ALYT}.deviation_supplementary"

# ---- Parse injected_context into structured entity arrays ----
def parse_injected_context(ctx: str) -> dict:
    """Parse the injected_context string into entity-type arrays."""
    result = {
        "clinical_ids": [],
        "documents": [],
        "acronyms": [],
        "cros": [],
        "devices": [],
        "vendors": [],
    }
    if not ctx:
        return result

    # Split on separator
    blocks = ctx.split("---")
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        # Identify entity type from the first line: [TYPE] "key"
        m = re.match(r'\[(\w+)\]\s*"([^"]+)"', block)
        if not m:
            continue
        etype = m.group(1).upper()
        key = m.group(2)

        # Extract canonical name
        canon_match = re.search(r'Canonical:\s*(.+)', block)
        canonical = canon_match.group(1).strip() if canon_match else key

        if etype == "CLINICAL_ID":
            result["clinical_ids"].append(canonical)
        elif etype == "DOCUMENT":
            result["documents"].append(canonical)
        elif etype == "ACRONYM":
            result["acronyms"].append(canonical)
        elif etype == "CRO":
            result["cros"].append(canonical)
        elif etype == "DEVICE":
            result["devices"].append(canonical)
        elif etype == "VENDOR":
            result["vendors"].append(canonical)

    return result

# Load and parse
embed_pdf = (
    spark.table(EMBED_INPUT)
    .select("pr_id", "injected_context")
    .toPandas()
)
embed_pdf["injected_context"] = embed_pdf["injected_context"].fillna("")
parsed = embed_pdf["injected_context"].apply(parse_injected_context)

entity_df = pd.DataFrame({
    "pr_id": embed_pdf["pr_id"],
    "clinical_ids": parsed.apply(lambda x: x["clinical_ids"]),
    "documents": parsed.apply(lambda x: x["documents"]),
    "acronyms": parsed.apply(lambda x: x["acronyms"]),
    "cros": parsed.apply(lambda x: x["cros"]),
    "devices": parsed.apply(lambda x: x["devices"]),
    "vendors": parsed.apply(lambda x: x["vendors"]),
})

print(f"Parsed {len(entity_df):,} events")
print(f"  with clinical_ids: {(entity_df['clinical_ids'].str.len() > 0).sum()}")
print(f"  with documents:    {(entity_df['documents'].str.len() > 0).sum()}")
print(f"  with acronyms:     {(entity_df['acronyms'].str.len() > 0).sum()}")
print(f"  with cros:         {(entity_df['cros'].str.len() > 0).sum()}")
print(f"  with devices:      {(entity_df['devices'].str.len() > 0).sum()}")
print(f"  with vendors:      {(entity_df['vendors'].str.len() > 0).sum()}")